# GSS 2018: synthpop versus LSM marginal distributions

Run cells in order with `gss_2018.csv` beside this notebook. Both generators fit the **same rows and variables**, and produce the same number of samples. By default this is an in-sample marginal fidelity comparison. Set `TRAIN_FRACTION = 0.5` for a shared training/holdout split. Frequencies are unweighted; exclude survey weights and identifiers explicitly if appropriate.

All columns are treated as **categorical survey codes**, including numeric-looking codes. Blank fields are a separate missing category; values such as NA, DK or refusal codes are preserved literally. This is unsuitable for continuous variables without discretization. Good marginals do not establish good joint/conditional structure or privacy.

Requirements: Python with pandas, numpy, matplotlib, nbformat, rpy2, lsynth and quasinet; R with synthpop. On MCC, select your existing Python environment as the kernel and load/configure R **before launching Jupyter**. Run expensive fits on an allocated compute node. No qmodel is required for synthpop or for these marginal metrics.

Source: https://github.com/zeroknowledgediscovery/lsynth (LSM uses `generate_syndata` and QuasiNet).

In [ ]:
# Optional installation: uncomment only if needed, then restart the kernel.
# %pip install pandas numpy matplotlib rpy2 lsynth quasinet
# Or, from the lsynth repository: %pip install -e .
# R installation (run in an R terminal):
# install.packages("synthpop", repos="https://cloud.r-project.org")

In [ ]:
from pathlib import Path
import json, time, random, hashlib, inspect
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

DATA_PATH = Path("gss_2018.csv")
OUT_DIR = Path("gss_2018_marginal_results")
N_SYNTH = 1000
SEED = 123
TRAIN_FRACTION = 1.0  # 0.5 = half for training, half for evaluation
N_WORKERS = 2
DROP_COLUMNS = []  # e.g. respondent ID and survey weight columns
MAX_LEVELS_EXCLUSIVE = 20  # keep only columns with strictly fewer than 20 levels
LSM_CSV = None  # Path("existing_lsm_gss_2018.csv") to skip LSM training
PLOT_COLUMNS = []  # empty = select 6 variables with largest synthpop-vs-LSM TV
MAX_PLOT_LEVELS = 15
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert 0 < TRAIN_FRACTION <= 1 and N_SYNTH > 0
random.seed(SEED)
np.random.seed(SEED)

## Load and encode consistently
Before splitting or fitting, keep only columns with **fewer than 20 distinct values** in the full input data (after explicit column drops). Empty strings count as a level, so a column with exactly 20 levels is excluded. The same filtered columns are used by both models and all marginal comparisons. This full-data schema filter also applies when using a holdout.

Short tokens prevent string truncation during lsynth sampling. The same reversible mapping is used for both models. Constant columns are restored after synthesis. Review the column inventory before proceeding.

In [ ]:
real = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False).drop(columns=DROP_COLUMNS)
assert len(real) >= 2 and real.shape[1] > 0
assert real.columns.is_unique
# Whitespace is intentionally preserved; only genuinely empty CSV fields mean missing.
inventory = pd.DataFrame({"levels": real.nunique(), "blank_fraction": real.eq("").mean()})
inventory["retained"] = inventory["levels"] < MAX_LEVELS_EXCLUSIVE
display(inventory)
inventory.to_csv(OUT_DIR / "column_inventory.csv")
excluded_columns = inventory.index[~inventory["retained"]].tolist()
print("Excluded columns (20 or more levels):", excluded_columns)
real = real.loc[:, inventory.index[inventory["retained"]]].copy()
if real.shape[1] == 0:
    raise ValueError("No columns have fewer than 20 levels")
print("Filtered data shape:", real.shape)
order = np.random.default_rng(SEED).permutation(len(real))
n_train = len(real) if TRAIN_FRACTION == 1 else int(len(real) * TRAIN_FRACTION)
assert n_train >= 2 and (TRAIN_FRACTION == 1 or n_train < len(real))
train = real.iloc[order[:n_train]].reset_index(drop=True)
reference = real if TRAIN_FRACTION == 1 else real.iloc[order[n_train:]].reset_index(drop=True)
reference_label = "original" if TRAIN_FRACTION == 1 else "holdout"
constants = {col: train[col].iloc[0] for col in train if train[col].nunique() == 1}
active = [col for col in train if col not in constants]
encoders = {col: {value: f"v{i}" for i, value in enumerate(pd.unique(train[col]))} for col in active}
decoders = {col: {token: value for value, token in mapping.items()} for col, mapping in encoders.items()}
encoded = pd.DataFrame({col: train[col].map(encoders[col]) for col in active})
print(f"Training: {len(train)}; reference: {len(reference)}; active: {len(active)}; constant: {len(constants)}")
train.to_csv(OUT_DIR / "training_data.csv", index=False)
reference.to_csv(OUT_DIR / "reference_data.csv", index=False)
with open(OUT_DIR / "encoding.json", "w") as f:
    json.dump({"encoders": encoders, "constants": constants}, f, indent=2)

def decode_generated(df):
    if list(df.columns) != active:
        raise ValueError("Generated columns/order do not match training features")
    result = pd.DataFrame(index=range(len(df)))
    for col in active:
        values = df[col].astype(str).reset_index(drop=True)
        unknown = set(values) - set(decoders[col])
        if unknown:
            raise ValueError(f"Unknown generated tokens in {col}: {list(unknown)[:5]}")
        result[col] = values.map(decoders[col])
    for col, value in constants.items():
        result[col] = value
    return result.loc[:, real.columns]

timings = {}

## Fit and generate with R synthpop
Every active variable is an R factor. CART synthesizes subsequent variables; the first is sampled. The R object is saved for inspection/reuse. A large level count can increase memory and runtime.

In [ ]:
if active:
    import rpy2.robjects as ro
    from rpy2.robjects import pandas2ri
    from rpy2.robjects.conversion import localconverter
    if not bool(ro.r('requireNamespace("synthpop", quietly=TRUE)')[0]):
        raise RuntimeError("Install R synthpop first: install.packages('synthpop', repos='https://cloud.r-project.org')")
    with localconverter(ro.default_converter + pandas2ri.converter):
        r_data = ro.conversion.py2rpy(encoded)
    synthesize = ro.r('''function(d, k, seed, outfile) {
        d[] <- lapply(d, factor)
        fit <- synthpop::syn(d, m=1, k=as.integer(k), seed=as.integer(seed),
                            method="cart", maxfaclevels=max(60L, max(vapply(d, nlevels, integer(1)))),
                            print.flag=FALSE)
        saveRDS(fit, outfile)
        result <- as.data.frame(lapply(fit$syn, as.character), check.names=FALSE)
        names(result) <- names(d)
        result
    }''')
    started = time.perf_counter()
    r_syn = synthesize(r_data, N_SYNTH, SEED, str((OUT_DIR / "synthpop_fit.rds").resolve()))
    timings["synthpop_fit_and_generate_seconds"] = time.perf_counter() - started
    with localconverter(ro.default_converter + pandas2ri.converter):
        syn_encoded = ro.conversion.rpy2py(r_syn)
    synthpop_df = decode_generated(syn_encoded)
else:
    synthpop_df = pd.DataFrame([constants] * N_SYNTH).loc[:, real.columns]
assert len(synthpop_df) == N_SYNTH
synthpop_df.to_csv(OUT_DIR / "synthpop_generated.csv", index=False)
print("synthpop:", synthpop_df.shape, timings)

## Fit and generate with LSM in lsynth
Default: train a fresh QuasiNet on exactly the synthpop training data. An existing LSM CSV must contain decoded original values and exactly the same filtered columns; you must verify that it used the same training cohort/preprocessing. Its first `N_SYNTH` rows are used, so supply at least that many. LSM threaded sampling may not be fully reproducible across environments.

In [ ]:
if LSM_CSV is not None:
    lsm_df = pd.read_csv(LSM_CSV, dtype=str, keep_default_na=False)
    if set(lsm_df.columns) != set(real.columns) or len(lsm_df) < N_SYNTH:
        raise ValueError("LSM CSV needs all original columns and at least N_SYNTH rows")
    lsm_df = lsm_df.loc[:, real.columns].iloc[:N_SYNTH].reset_index(drop=True)
elif active:
    from quasinet.qnet import Qnet, save_qnet
    from lsynth import generate_syndata
    if "model" not in inspect.signature(generate_syndata).parameters:
        raise RuntimeError("Update lsynth: this notebook requires generate_syndata(model=...)")
    started = time.perf_counter()
    model = Qnet(feature_names=active, n_jobs=N_WORKERS, alpha=0.05, random_state=SEED)
    model.fit(encoded.to_numpy(dtype=str))
    timings["lsm_training_seconds"] = time.perf_counter() - started
    save_qnet(model, str(OUT_DIR / "gss_2018_lsm.qmodel"), gz=True)
    started = time.perf_counter()
    np.random.seed(SEED)
    lsm_encoded = generate_syndata(num=N_SYNTH, model=model, orig_df=encoded,
                                 feature_names=active, gen_algorithm="LSM", n_workers=N_WORKERS)
    timings["lsm_generation_seconds"] = time.perf_counter() - started
    lsm_df = decode_generated(lsm_encoded)
else:
    lsm_df = pd.DataFrame([constants] * N_SYNTH).loc[:, real.columns]
assert len(lsm_df) == N_SYNTH
lsm_df.to_csv(OUT_DIR / "lsm_generated.csv", index=False)
print("LSM:", lsm_df.shape, timings)

## Compare every marginal
Total variation (TV) = half the sum of absolute differences in category probabilities. It ranges from 0 (identical) to 1 (disjoint). The union of categories from all datasets is included; empty strings count as a category. We report both synthetic-vs-reference distances and the direct synthpop-vs-LSM distance. Each variable has equal weight in summary averages. These are empirical estimates from a single run, not significance tests.

In [ ]:
def marginal_table(col):
    frames = {"reference": reference, "synthpop": synthpop_df, "lsm": lsm_df}
    levels = sorted(set().union(*(set(df[col]) for df in frames.values())))
    return pd.DataFrame({name: df[col].value_counts(normalize=True).reindex(levels, fill_value=0)
                         for name, df in frames.items()}).rename_axis("category")

def total_variation(p, q):
    return float(0.5 * np.abs(p - q).sum())

rows, long_tables = [], []
for col in real:
    tab = marginal_table(col)
    rows.append({"variable": col, "levels": len(tab),
                 "synthpop_vs_reference_tv": total_variation(tab.synthpop, tab.reference),
                 "lsm_vs_reference_tv": total_variation(tab.lsm, tab.reference),
                 "synthpop_vs_lsm_tv": total_variation(tab.synthpop, tab.lsm)})
    long_tables.append(tab.reset_index().assign(variable=col))
scores = pd.DataFrame(rows).set_index("variable")
scores["synthpop_minus_lsm_reference_tv"] = scores.synthpop_vs_reference_tv - scores.lsm_vs_reference_tv
scores = scores.sort_values("synthpop_vs_lsm_tv", ascending=False)
marginals = pd.concat(long_tables, ignore_index=True)[["variable", "category", "reference", "synthpop", "lsm"]]
scores.to_csv(OUT_DIR / "marginal_distances.csv")
marginals.to_csv(OUT_DIR / "marginal_probabilities.csv", index=False)
display(scores.head(20))
display(scores.filter(like="_tv").agg(["mean", "median", "max"]))

In [ ]:
# Frequency bars; omitted low-frequency levels are combined into one bar.
plot_cols = PLOT_COLUMNS or scores.head(6).index.tolist()
for i, col in enumerate(plot_cols):
    tab = marginal_table(col)
    if len(tab) > MAX_PLOT_LEVELS:
        keep = tab.max(axis=1).nlargest(MAX_PLOT_LEVELS - 1).index
        other = tab.loc[~tab.index.isin(keep)].sum()
        tab = tab.loc[keep].copy()
        tab.index = [repr(value) for value in tab.index]
        tab.loc["Other levels (combined)"] = other
    else:
        tab.index = ["<blank/missing>" if value == "" else value for value in tab.index]
    ax = tab.rename(columns={"reference": reference_label}).plot.bar(figsize=(11, 4), width=0.8)
    ax.set(title=col, ylabel="Proportion", xlabel="Category")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"marginal_{i + 1:02d}.png", dpi=160)
    plt.show()

ax = scores.plot.scatter(x="lsm_vs_reference_tv", y="synthpop_vs_reference_tv", figsize=(6, 6), alpha=0.6)
limit = max(0.01, float(scores[["lsm_vs_reference_tv", "synthpop_vs_reference_tv"]].max().max()) * 1.05)
ax.plot([0, limit], [0, limit], "--", color="gray")
ax.set(xlim=(0, limit), ylim=(0, limit), title="One point per variable",
       xlabel=f"LSM vs {reference_label}: TV", ylabel=f"synthpop vs {reference_label}: TV")
plt.tight_layout()
plt.savefig(OUT_DIR / "tv_comparison.png", dpi=160)
plt.show()

In [ ]:
versions = {}
for package in ["pandas", "numpy", "rpy2", "lsynth", "quasinet"]:
    try:
        versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        versions[package] = "not installed"
report = {"data_path": str(DATA_PATH), "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
          "seed": SEED, "train_fraction": TRAIN_FRACTION, "training_rows": len(train),
          "reference_rows": len(reference), "generated_rows_per_model": N_SYNTH,
          "variables": real.shape[1], "drop_columns": DROP_COLUMNS,
          "max_levels_exclusive": MAX_LEVELS_EXCLUSIVE, "excluded_level_columns": excluded_columns,
          "lsm_csv": str(LSM_CSV) if LSM_CSV is not None else None,
          "timings": timings, "versions": versions,
          "mean_tv": scores.filter(like="_tv").mean().to_dict()}
with open(OUT_DIR / "run_summary.json", "w") as f:
    json.dump(report, f, indent=2)
print("Results saved to:", OUT_DIR.resolve())

### Reading the results
Lower synthetic-vs-reference TV means closer marginal fidelity. Points above the diagonal have lower TV for LSM; below it, lower TV for synthpop. The direct synthpop-vs-LSM distance measures disagreement, not which model is more faithful. Full per-category probabilities are saved for every column, even when plots group levels. Repeat across seeds and use a shared holdout for a stronger benchmark. With a holdout, categories absent from training can appear in the reference and will contribute to the distance.